In [0]:
%run ../config/config

In [0]:
# Imports
import sys
import time
import os

sys.path.insert(0, str(project_path))
from utils.logger import MLOpsLogger, log_dataframe_info
from utils.month_delta import month_delta

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StringType

from utils.data_preparation.config import get_dataprep_config
from utils.data_preparation.fuente_hm_atraso_intra import FuenteHmAtrasoIntra
from utils.data_preparation.universo_implementacion import UniversoImplementacion

In [0]:
# Logger load preparation data
logger = MLOpsLogger(
    name='03_load_preparation_data',
    log_level='DEBUG' if DEBUG_MLOPS else 'INFO',
    log_dir=logs_path,
    is_job_run=True,
    job_context={
        'mes_vta': mes_vta,
        'environment': env,
        'notebook': '03_load_preparation_data'
    }
)

In [0]:
RUN_HISTORICAL = first_run
if RUN_HISTORICAL and num_meses_historicos > 0:
    meses_historicos_lista = []
    mes_tmp = mes_vta
    for i in range(num_meses_historicos):
        mes_tmp = month_delta(str(mes_tmp), -1)
        meses_historicos_lista.append(mes_tmp)
    meses_historicos_lista.reverse()
    meses_a_procesar = meses_historicos_lista + [mes_vta]
    logger.info(f"MODO HISTÓRICO: {len(meses_a_procesar)} meses: {meses_a_procesar}")
else:
    meses_a_procesar = [mes_vta]
    logger.info(f"Procesando únicamente el mes actual: {mes_vta}")

spark = SparkSession.builder.getOrCreate()
resultados = []

for idx_mes, mes_actual_proceso in enumerate(meses_a_procesar, 1):
    logger.info("")
    logger.info("=" * 80)
    logger.info(f"PROCESANDO MES {idx_mes}/{len(meses_a_procesar)}: {mes_actual_proceso}")
    logger.info("=" * 80)

    try:
        logger.log_stage_start('load_preparation_data', mes_vta=mes_actual_proceso, environment=env)
        start_time = time.time()

        # --- PREPARACIÓN DE DATOS REFACTORIZADA ---
        logger.info(f"Obteniendo configuración DataPrep para periodo={mes_actual_proceso}")
        dp_config = get_dataprep_config(periodo=str(mes_actual_proceso))

        logger.info(f"Verificando mora intrames para periodo={mes_actual_proceso}")
        req = [int(month_delta(str(mes_actual_proceso), -k)) for k in range(6)]
        path_hm = f"{catalog_name}.{schema_name}.{dp_config['sink_table_hm_atraso']}"
        try:
            existentes = {r[0] for r in spark.table(path_hm).select("codmes").distinct().collect()}
        except Exception:
            existentes = set()
        faltantes = sorted(set(req) - existentes)
        if faltantes:
            logger.info(f"  Meses faltantes en hm_atraso: {faltantes} — ejecutando FuenteHmAtrasoIntra")
            FuenteHmAtrasoIntra(
                spark=spark,
                codmes_ini=min(faltantes),
                codmes_fin=max(faltantes),
                src_catalog=dp_config["src_catalog"],
                sink_catalog=catalog_name,
                sink_schema=schema_name,
                sink_table_hm_atraso_cta=dp_config["sink_table_hm_atraso_cta"],
                sink_table_hm_atraso=dp_config["sink_table_hm_atraso"],
            ).execute()
        else:
            logger.info(f"  Mora intramés completa para los meses requeridos")

        logger.info(f"Ejecutando UniversoImplementacion para periodo={mes_actual_proceso}")
        universo_imp = UniversoImplementacion(
            spark=spark,
            codmes_ini=dp_config["codmes_ini"],
            codmes_fin=dp_config["codmes_fin"],
            src_catalog=dp_config["src_catalog"],
            src_schema_portafolio=dp_config["src_schema_portafolio"],
            src_table_portafolio=dp_config["src_table_portafolio"],
            # Utilizar configuración global (config.ipynb) para el destino troncal lógico
            sink_catalog=catalog_name,
            sink_schema=schema_name,
            sink_table_portafolio_troncal=BASE_NAME_TABLE_MASTER_ENV,
            sink_table_hm_atraso=dp_config["sink_table_hm_atraso"]
        )
        universo_imp.execute()
        # --------------------------------
        full_table_name_mt = TABLE_MASTER # OBS: esto no está amarrado a dp_config
        # Final count for logging
        count_final = spark.table(full_table_name_mt).count()
        logger.info(f"Registros finales en portafolio troncal ({full_table_name_mt}): {count_final:,}")

        total_duration = time.time() - start_time
        logger.log_stage_end('load_preparation_data', duration=total_duration, records_extracted=count_final)
        resultados.append({'mes': mes_actual_proceso, 'registros': count_final, 'exitoso': True, 'duracion': total_duration})

    except Exception as e:
        logger.log_exception(operation='load_preparation_data', exception=e, should_raise=False, mes_vta=mes_actual_proceso)
        resultados.append({'mes': mes_actual_proceso, 'registros': 0, 'exitoso': False, 'error': str(e)})
        continue

In [0]:
logger.info("\n" + "=" * 80)
logger.info("RESUMEN FINAL")
logger.info("=" * 80)
exitosos = sum(1 for r in resultados if r['exitoso'])
logger.info(f"Meses: {len(resultados)} | OK: {exitosos} | Error: {len(resultados) - exitosos}")

for r in resultados:
    if r['exitoso']:
        logger.info(f"  ✓ {r['mes']}: {r['registros']:,} registros en {r['duracion']:.2f}s")
    else:
        logger.error(f"  X {r['mes']}: {r.get('error', '?')}")

if 'logger' in locals():
    logger.info(f"Finalizando notebook: {logger.name}")
    logger._flush_all_handlers()
    logger.close()

#Validaciones

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, NumericType
import pandas as pd
import numpy as np
from pyspark.testing import assertSchemaEqual
pd.set_option('display.max_colwidth', None)

def compare_dataframes(
    df_base,
    df_target,
    cols_to_compare,
    cols_keys,
    threshold=1e-6,
    suffix_base="_base",
    suffix_target="_target"
):
    def suffix_cols(cols, suffix):
        return [F.col(c).alias(c + suffix) if c not in cols_keys else F.col(c) for c in cols]

    # Compare number of records in each dataframe
    df_base_count = df_base.count()
    df_target_count = df_target.count()
    if df_base_count != df_target_count:
        raise ValueError(f"Count mismatch: df_base={df_base_count}, df_target={df_target_count}")

    # Select & rename
    df_b = df_base.select(*suffix_cols(cols_to_compare + cols_keys, suffix_base))
    df_t = df_target.select(*suffix_cols(cols_to_compare + cols_keys, suffix_target))

    # Join
    df = df_b.join(df_t, cols_keys, "outer")

    # Build comparison expressions
    exprs = df.columns.copy()

    for c in cols_to_compare:
        l = F.col(c + suffix_base)
        r = F.col(c + suffix_target)

        ldt = df.select(l).schema[0].dataType
        rdt = df.select(r).schema[0].dataType

        if isinstance(ldt, StringType) and isinstance(rdt, StringType):
            exprs.append(
                F.when(l.isNull() & r.isNull(), 1)
                .when(l == r, 1)
                .otherwise(0)
                .alias(c + "_eq")
            )
        elif isinstance(ldt, NumericType) and isinstance(rdt, NumericType):
            exprs.extend([
                (l - r).alias(c + "_diff"),
                F.when(l.isNull() & r.isNull(), 1)
                .when(F.abs(l - r) <= threshold, 1)
                .otherwise(0)
                .alias(c + "_eq")
            ])

    df_cmp = df.select(*exprs).cache()
    # df_cmp.count()

    # Summary
    eq_cols = [c for c in df_cmp.columns if c.endswith("_eq")]

    ones = df_cmp.agg(*[F.sum(c).alias(c) for c in eq_cols]).withColumn("eq", F.lit(1))
    zeros = df_cmp.agg(*[(F.count("*") - F.sum(c)).alias(c) for c in eq_cols]).withColumn("eq", F.lit(0))

    df_summary = ones.unionByName(zeros).select("eq", *eq_cols)

    pd_summary = (
        df_summary
        .orderBy("eq")
        .toPandas()
        .set_index("eq")
        .T
        .reset_index()
        .rename(columns={"index": "var", 0: "count_0", 1: "count_1"})
        .sort_values("count_0", ascending=False)
    )

    pd_summary['porc_equiv'] = np.divide(pd_summary['count_1'], pd_summary['count_0'] + pd_summary['count_1'])
    pd_summary['porc_equiv'] = pd_summary['porc_equiv'].round(6)
    pd_summary['porc_equiv'] = pd_summary['porc_equiv'] * 100
    pd_summary['validacion'] = np.where(pd_summary['porc_equiv'] >= 99.0, '✅', '❌')

    return df_cmp, pd_summary


from pyspark.sql.types import StructType

def compare_schemas(schema1: StructType, schema2: StructType):
    fields1 = {f.name: f for f in schema1.fields}
    fields2 = {f.name: f for f in schema2.fields}
    all_cols = set(fields1.keys()).union(fields2.keys())

    diffs = []
    for col in sorted(all_cols):
        f1 = fields1.get(col)
        f2 = fields2.get(col)

        if f1 is None:
            diffs.append(f"+ Column only in target: {col} {f2.dataType}")
        elif f2 is None:
            diffs.append(f"- Column only in base: {col} {f1.dataType}")
        else:
            differences = []
            if f1.dataType != f2.dataType:
                differences.append(f"type: {f1.dataType} != {f2.dataType}")
            if f1.nullable != f2.nullable:
                differences.append(f"nullable: {f1.nullable} != {f2.nullable}")

            if differences:
                diffs.append(f"* Column '{col}' differs -> " + ", ".join(differences))

    return diffs

In [0]:
df_base = spark.table("catalog_cemm_expl_bcp_prod.bcp_expl_007.bhv_troncal_cliente_base_v2_dev")
df_target = spark.table("catalog_lhcl_prod_bcp_expl.bcp_edv_mmgr.T15724_bhv_port_d_mod_dev_gb_out_v3_history_vu_cor_copy")

# compare_schemas(df_base.schema, df_target.schema)

config_codmes_list = [202510, 202511, 202512]

cols_keys = ['codmes', 'codclaveunicocli']
vars_cols = [c for c in df_base.columns if c not in cols_keys + ["codclavepartycli", "codinternocomputacional"]]
for codmes in config_codmes_list:
    df_base_codmes = df_base.filter(F.col('codmes')==F.lit(codmes))
    df_target_codmes = df_target.filter(F.col('codmes')==F.lit(codmes))
    df_codmes, summary_codmes = compare_dataframes(
        df_base=df_base_codmes,
        df_target=df_target_codmes,
        cols_to_compare=vars_cols,
        cols_keys=cols_keys,
        threshold=1e-6,
        suffix_base="_base",
        suffix_target="_target"
    )
    print('='*60)
    print(codmes)
    print('-'*60)
    display(summary_codmes)

In [0]:
compare_schemas(df_base.schema, df_target.schema)

In [0]:
df_target.printSchema()
df_base.printSchema()

In [0]:
#len([c for c in df_base.columns if c in df_target.columns])
len([c for c in df_target.columns if c in df_base.columns])

In [0]:
# Descripción de ctdmora_intra_0_o por codmes en la tabla base
print("Tabla Base:")
df_base.groupBy("codmes") \
    .agg(
        F.count("ctdmora_intra_0_o").alias("count"),
        F.mean("ctdmora_intra_0_o").alias("mean"),
        F.stddev("ctdmora_intra_0_o").alias("stddev"),
        F.min("ctdmora_intra_0_o").alias("min"),
        F.max("ctdmora_intra_0_o").alias("max")
    ) \
    .orderBy("codmes") \
    .display()

# Descripción de ctdmora_intra_0_o por codmes en la tabla target
print("Tabla Target:")
df_target.filter(F.col("codmes") >= 202510) \
    .groupBy("codmes") \
    .agg(
        F.count("ctdmora_intra_0_o").alias("count"),
        F.mean("ctdmora_intra_0_o").alias("mean"),
        F.stddev("ctdmora_intra_0_o").alias("stddev"),
        F.min("ctdmora_intra_0_o").alias("min"),
        F.max("ctdmora_intra_0_o").alias("max")
    ) \
    .orderBy("codmes") \
    .display()

In [0]:
df_base.select("codmes").distinct().display()
df_target.select("codmes").distinct().display()

In [0]:
from pyspark.sql import functions as F

df_mio    = spark.table("catalog_cemm_expl_bcp_prod.bcp_expl_007.bhv_troncal_cliente_base_v2_dev").filter(F.col("codmes")==202512)
df_modelador = spark.table("catalog_lhcl_prod_bcp_expl.bcp_edv_mmgr.T15724_bhv_port_d_mod_dev_gb_out_v3_history_vu_cor_copy").filter(F.col("codmes")==202512)

# las 32 features del modelo (las que importan)
feats = [c for c in df_modelador.columns
            if c not in ("codmes", "codclaveunicocli", "codclavepartycli", "codinternocomputacional", "numpd", "numxb", "numscore")]

n_mio    = df_mio.count()
n_modelador = df_modelador.count()

print(f"{'variable':60s} {'%null_mio':>10s} {'%null_modelador':>13s}  dif")
print("="*95)
for c in feats:
    if c not in df_mio.columns:
        print(f"{c:60s} -- no existe en mi tabla --")
        continue
    nm = df_mio.filter(F.col(c).isNull()).count()
    ns = df_modelador.filter(F.col(c).isNull()).count()
    pm = 100*nm/n_mio
    ps = 100*ns/n_modelador
    flag = "  <-- DIFIERE" if abs(pm-ps) > 1.0 else ""
    print(f"{c:60s} {pm:10.2f} {ps:13.2f}{flag}")